In [0]:
from pyspark.sql.functions import count, max, min, avg, sum, round, col
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
two_months_ago_start = date.today().replace(day=1) - relativedelta(months=2)

In [0]:
df = spark.read.table("nyctaxi_02_silver.yellow_trips_enriched").filter(
    col("tpep_pickup_datetime") >= str(two_months_ago_start)
)

In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
nyctaxi_workspace,default


In [0]:
df = spark.read.table("nyctaxi_workspace.nyctaxi_02_silver.yellow_trips_enriched")

from pyspark.sql.functions import count, avg, max, min, sum, round

df = df.groupBy(df.tpep_pickup_datetime.cast("date").alias("pickup_date")).agg(
    count("*").alias("total_trips"),
    round(avg("passenger_count"), 1).alias("average_passengers"),
    round(avg("trip_distance"), 1).alias("average_distance"),
    round(avg("fare_amount"), 2).alias("average_fare_per_trip"),
    max("fare_amount").alias("max_fare"),
    min("fare_amount").alias("min_fare"),
    round(sum("total_amount"), 2).alias("total_revenue")
)
display(df)


pickup_date,total_trips,average_passengers,average_distance,average_fare_per_trip,max_fare,min_fare,total_revenue
2026-05-11,123429,1.2,4.4,21.02,1400.0,-240.3,3819608.59
2026-05-09,133671,1.3,4.3,19.27,500.0,-310.3,3649086.33
2026-05-31,120494,1.3,4.3,22.47,690.0,-229.1,3731411.13
2026-05-26,115659,1.2,4.9,21.43,518.2,-518.2,3523431.96
2026-05-22,119291,1.3,3.9,20.87,999.0,-170.0,3548919.76
2026-05-24,95737,1.4,9.4,19.87,895.0,-243.1,2671305.76
2026-05-30,142471,1.3,5.2,20.21,1755.8,-595.9,3990559.79
2026-05-23,116351,1.4,5.5,19.85,745.0,-170.3,3223809.36
2026-05-02,135716,1.3,4.5,19.95,850.0,-635.0,3811769.98
2026-05-25,78453,1.3,4.7,22.21,5525.99,-950.0,2484571.73


In [0]:
%sql
DROP TABLE IF EXISTS nyctaxi_workspace.nyctaxi_03_gold.daily_trip_summary;

In [0]:
df.write.mode("append").saveAsTable(
    "nyctaxi_workspace.nyctaxi_03_gold.daily_trip_summary"
)